In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from cmdstanpy import CmdStanModel

from lib.models.utils import (
    PerceptualModule,
    ResponseModule,
    get_stan_model_paths
)
from lib import filepaths

In [ ]:
stan_model_paths = get_stan_model_paths()

perceptual_map = dict(
    basic=PerceptualModule.BASIC,
    scpw=PerceptualModule.SCPW
)

response_map = dict(
    basic=ResponseModule.BASIC,
    brt=ResponseModule.BRT,
    brrt=ResponseModule.BRRT
)

In [ ]:
#--- PARAMETERS ---#
perceptual_model = "scpw"
response_model = "brt"
model_type = "2level"
hierarchy = "single_session"
n_chains = 3
n_samples = 1000
n_warmup = 200
#--- ---#

name_suffix = f"perc__{perceptual_model}-res__{response_model}"
output_dir = Path(filepaths.ROOT_STAN_MODEL_FITS)/hierarchy/model_type
os.makedirs(output_dir,exist_ok=True)

In [33]:
d = stan_model_paths[
    (stan_model_paths.perc == perceptual_map[perceptual_model]) 
    & 
    (stan_model_paths.resp == response_map[response_model])
]

stan_model_path = Path(d[f"{hierarchy}_root"].iloc[0])/model_type/d.filename.iloc[0]

model = CmdStanModel(stan_file=stan_model_path)

11:17:36 - cmdstanpy - INFO - compiling stan file /mnt/c/Users/marco/Desktop/research/ibl_gc/data/behavioral/stan_models/single_session/2level/perc__stimulus_contrast_precision_weight-res__belief_reliability_tradeoff.stan to exe file /mnt/c/Users/marco/Desktop/research/ibl_gc/data/behavioral/stan_models/single_session/2level/perc__stimulus_contrast_precision_weight-res__belief_reliability_tradeoff
11:17:44 - cmdstanpy - INFO - compiled model executable: /mnt/c/Users/marco/Desktop/research/ibl_gc/data/behavioral/stan_models/single_session/2level/perc__stimulus_contrast_precision_weight-res__belief_reliability_tradeoff


In [ ]:
def run_single_session_fit():
    # load full dataset
    dataset = pd.read_csv(filepaths.ROOT_BEHAV_DATA)
    
    # model fit for each session
    for session_id_code in dataset.session_id_code.unique():
        
        df_session = dataset[dataset.session_id_code==session_id_code]
        eid = df_session.session_id.unique()[0]
        
        data = dict(
            N_obs=len(df_session.stimulus_side),
            stimulus_side=df_session.stimulus_side.values,
            stimulus_contrast=df_session.stimulus_contrast.values,
            choice=df_session.choice.values,
            feedback=df_session.feedback
        )
        
        fit = model.sample(
            data=data, 
            chains=n_chains, 
            parallel_chains=n_chains, 
            iter_sampling=n_samples, 
            iter_warmup=n_warmup
        )
        
        savename = f"{eid}-{name_suffix}"
        np.save(output_dir/savename, fit)

In [ ]:
def run_single_subject_fit():
    # work on subjects with a sufficient number of sessions
    dataset = pd.read_csv(filepaths.ROOT_BEHAV_DATA)
    subj_ids_high_sessions = dataset.groupby("subj_id_code").session_id_code.nunique().reset_index().query("session_id_code >= 4").subj_id_code.values
    dataset = dataset[dataset.subj_id_code.isin(subj_ids_high_sessions)].reset_index(drop=True)
    subj_id_map = {s:i for i,s in enumerate(dataset.subj_id_code.unique())}
    session_id_map = {s:i for i,s in enumerate(dataset.session_id_code.unique())}
    dataset["subj_id_code"] = dataset.subj_id_code.apply(lambda x: subj_id_map[x])
    dataset["session_id_code"] = dataset.session_id_code.apply(lambda x: session_id_map[x])

    # model fit for each subject
    for subj_id_code in dataset.subj_id_code.unique():
        subj_id = dataset[dataset.subj_id_code==subj_id_code].subj_id.unique()[0]
        
        df_subj = dataset[dataset.subj_id==subj_id].reset_index()
        start_idx, end_idx = np.array([
            (idx.min() + 1, idx.max() + 1) 
            for _, idx in df_subj.groupby('session_id_code').groups.items()
        ]).T

        data = dict(
            N_obs=len(df_subj.stimulus_side),
            N_sess=df_subj.session_id_code.nunique(),
            stimulus_side=df_subj.stimulus_side.values,
            stimulus_contrast=df_subj.stimulus_contrast.values,
            feedback=df_subj.feedback.values,
            choice=df_subj.choice.values,
            start_idx=start_idx,
            end_idx=end_idx
        )
        
        fit = model.sample(
            data=data, 
            chains=n_chains, 
            parallel_chains=n_chains, 
            iter_sampling=n_samples, 
            iter_warmup=n_warmup
        )
        
        savename = f"{subj_id}-{name_suffix}"
        np.save(output_dir/savename, fit)

In [ ]:
def run_full_fit():
    # load full dataset
    dataset = pd.read_csv(filepaths.ROOT_BEHAV_DATA)
    
    # model fit
    start_idx, end_idx = np.array([
        (idx.min() + 1, idx.max() + 1) 
        for _, idx in dataset.groupby('session_id_code').groups.items()
    ]).T
    subj_ids, _ = dataset.groupby(["subj_id_code", "session_id_code"]).choice.mean().reset_index().iloc[:,:2].values.T
    
    data = dict(
        N_obs=len(dataset.stimulus_side),
        N_sess=dataset.session_id_code.nunique(),
        N_subj=dataset.subj_id_code.nunique(),
        subj_ids=subj_ids+1,
        stimulus_side=dataset.stimulus_side.values,
        stimulus_contrast=dataset.stimulus_contrast.values,
        feedback=dataset.feedback.values,
        choice=dataset.choice.values,
        start_idx=start_idx,
        end_idx=end_idx
    )
    
    fit = model.sample(
        data=data, 
        chains=n_chains, 
        parallel_chains=n_chains, 
        iter_sampling=n_samples, 
        iter_warmup=n_warmup
    )
    
    savename = f"full-{name_suffix}"
    np.save(output_dir/savename, fit)